# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/LaibaTaseen/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [11]:
%pip -q install duckdb huggingface_hub


In [12]:
import os, getpass

# Token order: env var -> Colab Secret -> prompt (last resort).
# Use a Colab Secret named HF_TOKEN (the key panel on the left) so the prompt never
# fires: if Colab reconnects while a getpass prompt is open, the kernel waits on it
# forever ('Resuming execution...') and you have to restart the runtime.
HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')


Paste your Hugging Face READ token (hf_...): ··········


In [13]:
import duckdb

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'dim_clients':                f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content':                f"read_parquet('{REL}/dim_content.parquet')",
    'fact_daily':                 f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    'fact_daily_sample':          f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')",
    'fact_query_90d':             f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

for name, src in TABLES.items():
    n = con.sql(f'SELECT COUNT(*) FROM {src}').fetchone()[0]
    print(f'{name:22} {n:>12,} rows')


dim_clients                     104 rows
dim_content                 519,606 rows


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

fact_daily               78,835,655 rows
fact_daily_sample        11,694,072 rows
fact_query_90d            2,414,248 rows


## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.
*

Each row in the dataset represents one website page on one specific day for a particular client. For this assignment, I used data from March 2026 instead of the most recent month because the latest month is kept aside as a test dataset and should not be used while building or evaluating the model.

In [14]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

check = con.sql(f"""
    SELECT MIN(report_date) AS min_date, MAX(report_date) AS max_date, COUNT(*) AS row_count
    FROM {TABLES['fact_daily']}
    WHERE report_date >= '2026-03-01' AND report_date < '2026-04-01'
""").df()
print(check)

# Prove the grain: check if (client, content, date) is unique — i.e. one row per page per day
grain_check = con.sql(f"""
    SELECT client_hash_id, content_hash_id, report_date, COUNT(*) AS n
    FROM {TABLES['fact_daily']}
    WHERE report_date >= '2026-03-01' AND report_date < '2026-04-01'
    GROUP BY 1, 2, 3
    HAVING COUNT(*) > 1
""").df()
print(f"Rows where (client, content, date) is NOT unique: {len(grain_check)}")


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

    min_date   max_date  row_count
0 2026-03-01 2026-03-31    9841378


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows where (client, content, date) is NOT unique: 0


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

###Feature fields:
I used impressions, clicks, and average search position to create features that help the model understand how each page is performing over time.
###Label field:
I compared the change in impressions between two time periods to classify each page as Growing, Declining, or Worth Review.
###Context fields:
The client ID, page ID, and date were only used to identify and organize the data. They were not used by the model to make predictions.
###Excluded data:
I did not use client names or search query text because they contain private information and were not needed for this project.

In [15]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [16]:
# Check the columns in fact_daily to find any boolean/availability flags

con.sql(f"DESCRIBE SELECT * FROM {TABLES['fact_daily']}").df()

,column_name,column_type,null,key,default,extra
0,report_date,DATE,YES,None,None,None
1,client_hash_id,VARCHAR,YES,None,None,None
2,content_hash_id,VARCHAR,YES,None,None,None
3,client_has_gsc,BOOLEAN,YES,None,None,None
4,client_has_ga4,BOOLEAN,YES,None,None,None
5,gsc_data_available,BOOLEAN,YES,None,None,None
6,ga4_data_available,BOOLEAN,YES,None,None,None
7,gsc_impressions,BIGINT,YES,None,None,None
8,gsc_clicks,BIGINT,YES,None,None,None
9,gsc_sum_position,BIGINT,YES,None,None,None


In [17]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


availability_check = con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) AS rows_with_gsc_data
    FROM {TABLES['fact_daily']}
    WHERE report_date >= '2026-03-01' AND report_date < '2026-04-01'
""").df()

print(availability_check)



FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   total_rows  rows_with_gsc_data
0     9841378             3611061


In [19]:
# Five features built from March 2026 (month=2026-03), each knowable at the decision moment
features_march = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions)                         AS total_impressions,
        SUM(gsc_clicks)                               AS total_clicks,
        AVG(gsc_avg_position)                         AS avg_position,
        STDDEV(gsc_avg_position)                      AS position_volatility,
        COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) AS days_with_data
    FROM {TABLES['fact_daily']}
    WHERE report_date >= '2026-03-01' AND report_date < '2026-04-01'
    GROUP BY 1, 2
    HAVING total_impressions >= 50
""").df()

print(f'{len(features_march):,} content items with features')
features_march.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

116,114 content items with features


,client_hash_id,content_hash_id,total_impressions,total_clicks,avg_position,position_volatility,days_with_data
0,client_62f4a7e64f5e0096,content_d0dff76c889de68f,181.0,0.0,5.147402,3.196582,29
1,client_62f4a7e64f5e0096,content_2e6360ad20fd7107,899.0,1.0,5.145765,5.860183,31
2,client_62f4a7e64f5e0096,content_65c50dfe9d87a585,3108.0,0.0,6.969536,2.286957,30
3,client_62f4a7e64f5e0096,content_d49a012dcb924e31,329.0,0.0,5.177774,2.109420,31
4,client_62f4a7e64f5e0096,content_614baf2af4330bd7,772.0,1.0,4.685335,1.141410,31



##Available when? for each feature
###Total Impressions:
The total number of times a page was shown in search results during the month. It only uses data that was already available.

###Total Clicks:
The total number of clicks the page received during the month, based only on past activity.

###Average Position:
The page's average search ranking during the month, calculated using only the rankings that had already been recorded.

###Position Volatility:
This shows how much the page's search ranking changed over time. A higher value means the ranking moved up and down more often.

###Days with Data:
The number of days during the month when the page had valid search data available.

In [20]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

features_march['good_ranking'] = (features_march['avg_position'] < 10).astype(int)

honest_features = ['total_impressions', 'total_clicks', 'position_volatility', 'days_with_data']
X = features_march[honest_features]
y = features_march['good_ranking']

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
model_honest = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1).fit(X_tr, y_tr)

honest_score = accuracy_score(y_te, model_honest.predict(X_te))
print(f"Honest score (no leak): {honest_score:.3f}")

Honest score (no leak): 0.874


In [21]:
features_march['leaky_avg_position'] = features_march['avg_position']

leaky_features = honest_features + ['leaky_avg_position']
X_leak = features_march[leaky_features]

X_tr_l, X_te_l, y_tr_l, y_te_l = train_test_split(X_leak, y, test_size=0.25, random_state=42, stratify=y)
model_leaky = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1).fit(X_tr_l, y_tr_l)

leaky_score = accuracy_score(y_te_l, model_leaky.predict(X_te_l))
print(f"Leaky score (with label-derived column): {leaky_score:.3f}")
print(f"Jump: {leaky_score - honest_score:.3f}")

Leaky score (with label-derived column): 1.000
Jump: 0.126


The leaky feature gave the model a perfect score of 1.000, which showed that it was using information it should not have had. This happened because the feature was directly related to the answer the model was trying to predict. To keep the results fair, I removed this feature from the model. After removing it, the model achieved an accuracy of 0.874, which is a more realistic and trustworthy result.

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

###Idea 1: Limited Data

Not all of the available data could be used. Only about 37% of the March data contained real search information. The rest either had no tracking or no search activity, so the model learned from only the data that was available.

###Idea 2: The Data Doesn't Explain the Reason

The data shows what happened, such as impressions increasing or decreasing, but it doesn't explain why it happened. A page's performance could change for many reasons, like content quality, competition, seasonal trends, or technical issues.

###Idea 3: Different Amounts of Data for Each Client

Some clients have been tracked for a longer time than others. This means some clients have more data available, while others have less, so the dataset is not completely balanced.

In [18]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [X] Every section above is filled — markdown thinking AND the code that backs it
- [X] The notebook runs top to bottom with no errors (Runtime → Run all)
- [X] No client names, URLs, or private queries anywhere
- [X] My claims use careful words: observed, measured, directional, decision-support
- [X] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.